## Preprocessing
Normalize FITS images and create augmented copies (rotations + flips).

In [ ]:
import os, glob
import numpy as np
from astropy.io import fits
from scipy import ndimage
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

input_folder      = "train_resized"
normalized_folder = "train_normalized"
augmented_folder  = "train_augmented"

os.makedirs(normalized_folder, exist_ok=True)
os.makedirs(augmented_folder,  exist_ok=True)


In [ ]:
def normalize_data(data, method="minmax"):
    data = np.nan_to_num(data, 0)
    if method == "minmax":
        lo, hi = np.percentile(data, [1, 99])
        return np.clip((data - lo) / (hi - lo + 1e-10), 0, 1)
    if method == "zscore":
        n = np.clip((data - data.mean()) / (data.std() + 1e-10), -3, 3)
        return (n + 3) / 6
    if method == "percentile":
        p2, p98 = np.percentile(data, [2, 98])
        return np.clip((data - p2) / (p98 - p2 + 1e-10), 0, 1)

def augment_image(data):
    result = []
    for angle in [0, 90, 180, 270]:
        rot = ndimage.rotate(data, angle, reshape=False, order=1)
        result.append((f'rot{angle}',      rot))
        result.append((f'rot{angle}_flip', np.fliplr(rot)))
    return result


In [ ]:
# Compare all three normalization methods on every 10th file
methods = ["minmax", "zscore", "percentile"]
fits_files = sorted(glob.glob(os.path.join(input_folder, "*.fits")))

for i, fpath in enumerate(fits_files, 1):
    if i % 10 != 0:
        continue
    fname = os.path.basename(fpath)
    with fits.open(fpath) as hdul:
        data = hdul[0].data.copy()

    fig, axes = plt.subplots(1, len(methods)+1, figsize=(4*(len(methods)+1), 4))
    lo, hi = np.percentile(data, [1, 99])
    axes[0].imshow(data, cmap='gray', vmin=lo, vmax=hi, origin='lower')
    axes[0].set_title(f'Original\n{fname}'); axes[0].axis('off')
    for col, m in enumerate(methods, 1):
        axes[col].imshow(normalize_data(data, m), cmap='gray', origin='lower')
        axes[col].set_title(m); axes[col].axis('off')
    plt.tight_layout()
    plt.savefig(os.path.join(normalized_folder, f"cmp_{i:04d}.png"), dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Saved comparison for file #{i}: {fname}")


In [ ]:
# Pick minmax (best visual result) and run normalization + augmentation
normalize_method = "minmax"
print(f"Using: {normalize_method}")

norm_files = []
for i, fpath in enumerate(fits_files, 1):
    fname = os.path.basename(fpath)
    base  = os.path.splitext(fname)[0]
    with fits.open(fpath) as hdul:
        data = hdul[0].data
        hdr  = hdul[0].header
    normed = normalize_data(data, normalize_method)
    hdr['NORMTYPE'] = normalize_method
    out = os.path.join(normalized_folder, f"norm_{base}.fits")
    fits.PrimaryHDU(normed.astype(np.float32), hdr).writeto(out, overwrite=True)
    norm_files.append(out)
    if i % 10 == 0 or i == len(fits_files):
        print(f"  {i}/{len(fits_files)} normalised")

print(f"\nNormalization done: {len(norm_files)} files")


In [ ]:
# Augmentation
print("\nStarting augmentation...")
total_aug = 0

for i, fpath in enumerate(norm_files, 1):
    fname = os.path.basename(fpath)
    base  = os.path.splitext(fname)[0].replace('norm_', '')
    with fits.open(fpath) as hdul:
        data = hdul[0].data
        hdr  = hdul[0].header
    for suffix, aug_data in augment_image(data):
        aug_path = os.path.join(augmented_folder, f"aug_{base}_{suffix}.fits")
        fits.PrimaryHDU(aug_data.astype(np.float32), hdr).writeto(aug_path, overwrite=True)
        total_aug += 1
    if i % 10 == 0 or i == len(norm_files):
        print(f"  {i}/{len(norm_files)} done")

print(f"\nFinished!  Normalised: {len(norm_files)}  Augmented: {total_aug}")
